## 7. Conclusion

We've demonstrated a complete RAG pipeline using the latest `azure-ai-projects` SDK:

1. **Embed** documents with `text-embedding-3-small` via `AzureOpenAI` (built directly from the project endpoint)
2. **Store** them in Azure AI Search (HNSW vector index)
3. **Retrieve** relevant chunks using `VectorizedQuery`
4. **Generate** grounded answers with `gpt-4o`

**Next steps to explore:**
- Add `search_text` to `search_client.search()` for **hybrid search** (vector + BM25)
- Add a `$filter` on `source` field to restrict retrieval by category
- Swap to chunked PDFs / larger corpora
- Evaluate answer quality with `azure-ai-evaluation`


## 0. Install Dependencies

In [ ]:
# Option A — Latest stable (recommended)
%pip install --quiet --upgrade \
    azure-ai-projects \
    azure-identity \
    azure-search-documents \
    openai

# Option B — Latest including beta/pre-release features
# %pip install --quiet --upgrade --pre \
#     azure-ai-projects \
#     ...

## 1. Setup — Load Config & Initialize Clients

In [ ]:
import os
import json
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ConnectionType          # ← moved here in v1.0.0b5+
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential

# ------------------------------------------------------------------
# Utility: walk up directories to find cred.json
# ------------------------------------------------------------------
def find_cred_json(start_path):
    current = Path(start_path)
    while current != current.parent:
        cred_file = current / "cred.json"
        if cred_file.exists():
            return str(cred_file)
        current = current.parent
    return None

# ------------------------------------------------------------------
# Load configuration from cred.json
# ------------------------------------------------------------------
file_path = find_cred_json(os.getcwd())
if not file_path:
    raise FileNotFoundError("cred.json not found in current or parent directories")

with open(file_path, "r") as f:
    cfg = json.load(f)

endpoint                 = cfg.get("PROJECT_ENDPOINT")                              # https://<resource>.services.ai.azure.com/api/projects/<project>
model_deployment_name    = cfg.get("MODEL_DEPLOYMENT_NAME",            "gpt-4o")
embedding_model_name     = cfg.get("EMBEDDING_MODEL_DEPLOYMENT_NAME",  "text-embedding-3-small")
search_service_endpoint  = cfg.get("SEARCH_ENDPOINT")                               # https://<service>.search.windows.net
search_api_key           = cfg.get("SEARCH_API_KEY")
search_index_name        = cfg.get("SEARCH_INDEX_NAME",                "psitronaiserach")

print("Project Endpoint  :", endpoint)
print("Chat Model        :", model_deployment_name)
print("Embedding Model   :", embedding_model_name)
print("Search Index Name :", search_index_name)


In [ ]:
# ------------------------------------------------------------------
# Initialize AIProjectClient
# ------------------------------------------------------------------
credential     = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)

# ------------------------------------------------------------------
# Build an AzureOpenAI client directly — works on ALL versions of
# azure-ai-projects (1.0.0b*, 1.0.0 stable, 2.0.0b*).
#
# Why not use project_client helpers?
#   • get_openai_client()               → Responses API (/openai/v1),
#                                          no /embeddings route → 404
#   • inference.get_azure_openai_client()→ only in beta builds;
#                                          removed in 1.0.0 stable
#
# Solution: derive the Azure OpenAI endpoint from PROJECT_ENDPOINT
#   https://<resource>.services.ai.azure.com/api/projects/<name>
#   →  https://<resource>.services.ai.azure.com   (valid AOAI base)
# ------------------------------------------------------------------
from azure.identity import get_bearer_token_provider
from openai import AzureOpenAI

# Strip /api/projects/... to get the bare services endpoint
aoai_endpoint = endpoint.split("/api/projects")[0]   # e.g. https://resource.services.ai.azure.com

token_provider = get_bearer_token_provider(
    credential, "https://cognitiveservices.azure.com/.default"
)

openai_client = AzureOpenAI(
    azure_endpoint=aoai_endpoint,
    azure_ad_token_provider=token_provider,
    api_version="2024-06-01",
)

print("✅ AIProjectClient    :", type(project_client).__name__)
print("✅ AzureOpenAI client :", type(openai_client).__name__)
print("   Endpoint           :", aoai_endpoint)


In [ ]:
# ------------------------------------------------------------------
# Helper wrappers
# ------------------------------------------------------------------
def chat(prompt: str) -> str:
    response = openai_client.chat.completions.create(
        model=model_deployment_name,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

def embed_text(text: str) -> list:
    response = openai_client.embeddings.create(
        model=embedding_model_name,
        input=text
    )
    return response.data[0].embedding

# Quick smoke test
print("💬 Chat        :", chat("Hello, how are you?"))
print("📐 Embed length:", len(embed_text("MLOps is awesome")))

## 2. Create Sample Health Data

In [ ]:
health_tips = [
    {"id": "doc1", "content": "Daily 30-minute walks help maintain a healthy weight and reduce stress.",            "source": "General Fitness"},
    {"id": "doc2", "content": "Stay hydrated by drinking 8-10 cups of water per day.",                             "source": "General Fitness"},
    {"id": "doc3", "content": "Consistent sleep patterns (7-9 hours) improve muscle recovery.",                    "source": "General Fitness"},
    {"id": "doc4", "content": "For cardio endurance, try interval training like HIIT.",                             "source": "Workout Advice"},
    {"id": "doc5", "content": "Warm up with dynamic stretches before running to reduce injury risk.",               "source": "Workout Advice"},
    {"id": "doc6", "content": "Balanced diets include protein, whole grains, fruits, vegetables, and healthy fats.", "source": "Nutrition"},
]

print(f"✅ {len(health_tips)} health tips loaded.")
for tip in health_tips:
    print(f"  [{tip['id']}] ({tip['source']}) {tip['content']}")

## 3. Create or Reset the Search Index

Defines an HNSW vector search index with cosine similarity.
The `source` field is now `filterable=True` so you can filter by category later.

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchAlgorithmKind,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
)

def create_healthtips_index(
    endpoint: str,
    api_key: str,
    index_name: str,
    dimension: int = 1536,
):
    """Create (or reset) a vector-search-capable index for health tips."""
    index_client = SearchIndexClient(
        endpoint=endpoint,
        credential=AzureKeyCredential(api_key)
    )

    # Drop existing index if present
    try:
        index_client.delete_index(index_name)
        print(f"🗑  Deleted existing index: {index_name}")
    except Exception:
        pass

    vector_search = VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="myHnsw",
                kind=VectorSearchAlgorithmKind.HNSW,
                parameters=HnswParameters(
                    m=4,
                    ef_construction=400,
                    ef_search=500,
                    metric=VectorSearchAlgorithmMetric.COSINE,
                ),
            )
        ],
        profiles=[
            VectorSearchProfile(
                name="myHnswProfile",
                algorithm_configuration_name="myHnsw",
            )
        ],
    )

    fields = [
        SimpleField(name="id",      type=SearchFieldDataType.String, key=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SimpleField(name="source",  type=SearchFieldDataType.String, filterable=True),  # filterable for category queries
        SearchField(
            name="embedding",
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            vector_search_dimensions=dimension,
            vector_search_profile_name="myHnswProfile",
        ),
    ]

    index_client.create_index(
        SearchIndex(name=index_name, fields=fields, vector_search=vector_search)
    )
    print(f"✅ Created index: {index_name}  (dim={dimension})")

## 3.1 Verify Connection → Create Index → Embed & Upload

In [ ]:
# Step 1: Verify the default AI Search connection exists in your Foundry project
search_conn = project_client.connections.get_default(
    connection_type=ConnectionType.AZURE_AI_SEARCH,
    include_credentials=True,
)
if not search_conn:
    raise RuntimeError('❌ No default Azure AI Search connection found in Foundry project.')

# .target in beta → .endpoint_url in stable 1.0.0
conn_target = getattr(search_conn, 'target',
               getattr(search_conn, 'endpoint_url', 'n/a'))

print('✅ Search connection :', search_conn.name)
print('   Target endpoint  :', conn_target)


In [ ]:
# Step 2: Resolve embedding dimension dynamically
# (so swapping to text-embedding-3-large just works)
sample_emb    = openai_client.embeddings.create(model=embedding_model_name, input=health_tips[0]["content"])
embedding_dim = len(sample_emb.data[0].embedding)
print(f"✅ Embedding dimension: {embedding_dim}")

In [ ]:
# Step 3: Create (or reset) the index
create_healthtips_index(
    endpoint=search_service_endpoint,
    api_key=search_api_key,
    index_name=search_index_name,
    dimension=embedding_dim,
)

In [ ]:
# Step 4: SearchClient for uploads & queries
search_client = SearchClient(
    endpoint=search_service_endpoint,
    index_name=search_index_name,
    credential=AzureKeyCredential(search_api_key),
)
print("✅ SearchClient ready")

In [ ]:
# Step 5: Embed each doc and bulk-upload to the index
search_docs = []
for doc in health_tips:
    emb_resp = openai_client.embeddings.create(
        model=embedding_model_name,
        input=doc["content"],
    )
    search_docs.append({
        "id":        doc["id"],
        "content":   doc["content"],
        "source":    doc["source"],
        "embedding": emb_resp.data[0].embedding,
    })
    print(f"  Embedded {doc['id']}")

result = search_client.upload_documents(documents=search_docs)
print(f"\n✅ Uploaded {len(search_docs)} docs to '{search_index_name}'")

## 4. RAG Flow — Retrieve + Generate

### How it works
1. **Embed** the user question
2. **Vector search** the index for the top-k nearest docs
3. **Ground** the LLM response in those docs only

In [ ]:
from azure.search.documents.models import VectorizedQuery

def rag_chat(query: str, top_k: int = 3) -> str:
    # ── 1. Embed the user query ────────────────────────────────────
    q_emb = openai_client.embeddings.create(
        model=embedding_model_name,
        input=query,
    ).data[0].embedding

    # ── 2. Vector search ──────────────────────────────────────────
    results = search_client.search(
        search_text="",           # empty = pure vector; add text for hybrid
        vector_queries=[
            VectorizedQuery(
                vector=q_emb,
                k_nearest_neighbors=top_k,
                fields="embedding",
            )
        ],
        select=["content", "source"],
    )

    top_docs = [f"[{r['source']}] {r['content']}" for r in results]

    # ── 3. Grounded generation ────────────────────────────────────
    system_prompt = (
        "You are a health & fitness assistant.\n"
        "Answer ONLY using the documents below. If unsure, say 'I\'m not sure'.\n\n"
        "Documents:\n" + "\n".join(top_docs)
    )

    response = openai_client.chat.completions.create(
        model=model_deployment_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": query},
        ],
    )
    return response.choices[0].message.content

print("✅ rag_chat() defined")

## 5. Try Some Queries 🎉

In [ ]:
queries = [
    "What's a good short cardio routine for me if I'm busy?",
    "How much sleep should I get for recovery?",
    "What should I eat to stay healthy?",
]

for q in queries:
    print(f"🗣️  {q}")
    print(f"🤖  {rag_chat(q)}")
    print("-" * 60)

## 6. (Debug) List All Foundry Connections

In [ ]:
# The Connection object's attribute names changed between beta and stable 1.0.0.
# Use getattr() fallbacks so this works on any version.
print('All connections in this Foundry project:')
for conn in project_client.connections.list():
    name    = getattr(conn, 'name', 'n/a')
    # 'connection_type' in beta → 'type' in stable 1.0.0
    ctype   = getattr(conn, 'connection_type',
              getattr(conn, 'type', 'n/a'))
    # 'target' in beta → 'endpoint_url' or inside properties in stable
    target  = getattr(conn, 'target',
              getattr(conn, 'endpoint_url', 'n/a'))
    print(f'  • {name}  type={ctype}  target={target}')

# If you want to see ALL available attributes for your version:
# first_conn = next(iter(project_client.connections.list()), None)
# if first_conn: print(vars(first_conn))


## 7. Conclusion

We've demonstrated a complete RAG pipeline using the latest `azure-ai-projects` SDK:

1. **Embed** documents with `text-embedding-3-small` via `AzureOpenAI` (built directly from the project endpoint)
2. **Store** them in Azure AI Search (HNSW vector index)
3. **Retrieve** relevant chunks using `VectorizedQuery`
4. **Generate** grounded answers with `gpt-4o`

**Next steps to explore:**
- Add `search_text` to `search_client.search()` for **hybrid search** (vector + BM25)
- Add a `$filter` on `source` field to restrict retrieval by category
- Swap to chunked PDFs / larger corpora
- Evaluate answer quality with `azure-ai-evaluation`
